In [13]:
#Lab 5.2 — Mini-Transformer on Toy Language Task
#1. Toy dataset
import torch

data = torch.randint(0, 20, (200, 10))  # 200 sequences of length 10

labels = (data.sum(1) % 2).long()

train_x, test_x = data[:150], data[150:]
train_y, test_y = labels[:150], labels[150:]

print("Dataset shape:", data.shape)
print("Labels shape:", labels.shape)

print("\nFirst 5 sequences:")
print(data[:5])

print("\nFirst 5 labels:")
print(labels[:5])

print("\nTraining data:", train_x.shape)
print("Testing data:", test_x.shape)
print("Training labels:", train_y.shape)

Dataset shape: torch.Size([200, 10])
Labels shape: torch.Size([200])

First 5 sequences:
tensor([[ 3,  4, 12, 17,  8, 18, 16,  8,  4,  8],
        [16,  0,  0,  1, 18, 17, 18, 10, 10, 14],
        [ 7, 12,  4,  4, 12, 11, 10, 12, 18, 18],
        [ 7,  0,  7, 12,  6, 11,  5, 19, 16,  8],
        [12, 15, 19,  6,  5, 19,  2,  5, 17, 19]])

First 5 labels:
tensor([0, 0, 0, 1, 1])

Training data: torch.Size([150, 10])
Testing data: torch.Size([50, 10])
Training labels: torch.Size([150])


In [14]:
#2. Embedding + Transformer encoder
import torch
import torch.nn as nn

class TinyTransformer(nn.Module):

    def __init__(self, vocab=20, d_model=32, nhead=4, num_layers=2):
        super().__init__()

        self.emb = nn.Embedding(vocab, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True
        )

        self.trans = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.fc = nn.Linear(d_model, 2)

    def forward(self, x):
        h = self.emb(x)
        h = self.trans(h)
        return self.fc(h.mean(1))


model = TinyTransformer()

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
model = TinyTransformer()

print(model)
print("\nModel created successfully!")

TinyTransformer(
  (emb): Embedding(20, 32)
  (trans): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=32, out_features=32, bias=True)
        )
        (linear1): Linear(in_features=32, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=32, bias=True)
        (norm1): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (fc): Linear(in_features=32, out_features=2, bias=True)
)

Model created successfully!


In [15]:
#3. Training
import torch.nn.functional as F

for ep in range(10):

    model.train()

    out = model(train_x)

    loss = F.cross_entropy(out, train_y)

    opt.zero_grad()
    loss.backward()
    opt.step()

    print("Epoch", ep + 1, "Loss", loss.item())

Epoch 1 Loss 0.6930498480796814
Epoch 2 Loss 0.7004915475845337
Epoch 3 Loss 0.6978026628494263
Epoch 4 Loss 0.6908214688301086
Epoch 5 Loss 0.6976053714752197
Epoch 6 Loss 0.6996820569038391
Epoch 7 Loss 0.6968796849250793
Epoch 8 Loss 0.6973799467086792
Epoch 9 Loss 0.6974911689758301
Epoch 10 Loss 0.6961090564727783


In [16]:
#4. Evaluate
model.eval()

with torch.no_grad():

    predictions = model(test_x).argmax(1)

    acc = (predictions == test_y).float().mean()

print("Actual labels:    ", test_y)
print("Predicted labels: ", predictions)

print("Test accuracy:", acc.item())
print("Test accuracy (%):", acc.item() * 100)

Actual labels:     tensor([0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0,
        0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1,
        1, 1])
Predicted labels:  tensor([1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1,
        0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1,
        1, 0])
Test accuracy: 0.5199999809265137
Test accuracy (%): 51.99999809265137
